# pDockQ2 Interface Quality

![pDockQ2 Interface Quality](https://proto-bio.github.io/proto-assets/images/tool/pdockq2/hero.png)

This notebook demonstrates `run_pdockq2`, which scores the interface of a cofolded protein complex using the Zhu-2023 pDockQ2 formula. It combines per-residue pLDDT and the PAE matrix into a single scalar in `[0, 1]`. Germinal's final-filter stage rejects trajectories with `pdockq2 <= 0.23`.

Reference: Zhu et al., *Bioinformatics* 39:btad424 (2023). [doi:10.1093/bioinformatics/btad424](https://doi.org/10.1093/bioinformatics/btad424).

## Imports

The tool input requires a `Structure` whose B-factor column carries pLDDT (`b_factor_type=BFactorType.PLDDT` or `NORMALIZED_PLDDT`) and whose `metrics['pae']` contains the square PAE matrix emitted by the structure predictor.

In [1]:
from proto_tools import PDockQ2Config, PDockQ2Input, PDockQ2Output, run_pdockq2
from proto_tools.tools.structure_scoring.pdockq2.pdockq2 import example_input
from proto_tools.utils.notebook_docs import display_api_reference

## Input API Reference

In [2]:
display_api_reference("pdockq2", "input")

**Input** — `PDockQ2Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structure</code> | <code>Structure</code> | required | Cofolded complex with pLDDT in B-factors and the PAE matrix at metrics['pae'] |
| <code>binder_chain</code> | <code>SingleChainSelection</code> | required | Single-character chain ID of the binder |
| <code>target_chains</code> | <code>ChainSelection</code> | required | Target chain ID(s) |

## Config API Reference

In [3]:
display_api_reference("pdockq2", "config")

**Config** — `PDockQ2Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>distance_cutoff</code> | <code>float</code> | <code>10.0</code> | CA-CA distance cutoff (Å) for interface residue detection. |

## Output API Reference

In [4]:
display_api_reference("pdockq2", "output")

**Output** — `PDockQ2Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>metrics</code> | <code>PDockQ2Metrics</code> | required | pDockQ2 metrics for the input complex |

**Metrics** — `PDockQ2Metrics`

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>pdockq2</code> **(primary)** | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>avg_interface_plddt</code> | <code>float</code> | <code>[0, 100]</code> |  |  |
| <code>avg_interface_pae</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>num_interface_contacts</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |

## Basic Usage

`example_input()` returns a bundled **barnase–barstar** complex (chains A/B), cofolded by Boltz-2 with its real PAE matrix, so the notebook runs offline. In real usage, the `Structure` and `metrics['pae']` come from AF3 / Chai-1 / Boltz-2 / Protenix output.

In [5]:
inputs = example_input()
config = PDockQ2Config()
result: PDockQ2Output = run_pdockq2(inputs, config)

print(f"pdockq2:               {result.metrics.pdockq2:.4f}")
print(f"avg_interface_plddt:   {result.metrics.avg_interface_plddt:.2f}")
print(f"avg_interface_pae:     {result.metrics.avg_interface_pae:.4f}")
print(f"num_interface_contacts: {result.metrics.num_interface_contacts}")

pdockq2:               0.9404
avg_interface_plddt:   97.92
avg_interface_pae:     0.9903
num_interface_contacts: 100


## Per-Chain Interface Breakdown

The `interfaces` field exposes every chain's contribution, including those not aggregated into the final score. Useful when debugging multi-chain targets.

In [6]:
for row in result.metrics.interfaces:
    print(
        f"chain {row.chain_id} | neighbors={row.neighbor_chains!r:5} | "
        f"if_plddt={row.if_plddt:6.2f} | norm_pae={row.norm_pae:.4f} | "
        f"pmidockq={row.pmidockq:.4f}"
    )

chain A | neighbors='B'   | if_plddt= 96.99 | norm_pae=0.9887 | pmidockq=0.9186
chain B | neighbors='A'   | if_plddt= 97.92 | norm_pae=0.9903 | pmidockq=0.9404


## Tightening the Distance Cutoff

The distance cutoff controls which CA-CA pairs count as "in contact". Tightening it drops marginal pairs and, if no contacts remain, the score falls to zero.

In [7]:
tight = run_pdockq2(inputs, PDockQ2Config(distance_cutoff=6.0))
print(f"tight cutoff: pdockq2={tight.metrics.pdockq2:.4f}, contacts={tight.metrics.num_interface_contacts}")

tight cutoff: pdockq2=0.9516, contacts=7


## Export

Results can be exported as JSON for downstream analysis.

In [8]:
from pathlib import Path

out_dir = Path("./example_output")
out_dir.mkdir(exist_ok=True)
result.export("pdockq2_result", export_path=out_dir, file_format="json")
print(f"Exported to {out_dir / 'pdockq2_result.json'}")

Exported to example_output/pdockq2_result.json
